# Setup

In [1]:
# user setup
import os

# custom Kaggle login (environment variables are recommended)
os.environ["KAGGLE_USERNAME"] = "" or os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = "" or os.getenv("KAGGLE_KEY")

# spark configuration
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" or os.getenv("JAVA_HOME")
os.environ["SPARK_HOME"] = "" or os.getenv("SPARK_HOME")

# configuration
SEED = 25 # seed for reproducibility (None for random seed)
SAMPLE_SIZE = 1.0 # fraction of the dataset to be used for training (between 0 and 1)
MIN_KW_FREQ = 5 # minimum frequency of keywords to be included in the profile (for the dictionary approach)
FEATURES_SIZE = 5000 # number of features, buckets of the hash (for the hashing approach)
LSH_NUM_SKETCHES = 100 # number of random hyperplanes (signatures/sketches)
LSH_BANDS = 10 # TODO: play with b and r (calculate the threshold for different values)
LSH_ROWS = 10
LSH_MAX_BUCKET_SIZE = 500 # maximum number of users/articles in the same bucket
RECOMMENDED_ARTICLES = 10 # number of articles to recommend to each user
assert 0 < SAMPLE_SIZE <= 1, "invalid SAMPLE_SIZE, must be between 0 and 1"
assert LSH_NUM_SKETCHES == LSH_BANDS * LSH_ROWS, "invalid LSH_NUM_SKETCHES, must be equal to LSH_BANDS * LSH_ROWS"


In [2]:
# setup dataset
%pip install kaggle
!test -d dataset && echo "Skipping dataset download" || (kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020" && unzip -d dataset new-york-times-articles-comments-2020.zip && rm -f new-york-times-articles-comments-2020.zip 2> /dev/null)


Note: you may need to restart the kernel to use updated packages.
Skipping dataset download


In [ ]:
# setup spark
!test -d spark && echo "Skipping spark download" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1 > /dev/null && rm spark.tgz)
%pip install pyspark
%pip install py4j
import pyspark
ss = (pyspark.sql.SparkSession
    .builder
    .getOrCreate())
sc = ss.sparkContext


Skipping spark download
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
csv_options = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',
    "escape": '"', # quotes " in the dataset are escaped as ""
    "multiLine": "true",
    "mode": "PERMISSIVE",
}

full_articles = (ss
                 .read
                 .options(**csv_options)
                 .csv("dataset/nyt-articles-2020.csv")
                 .sample(withReplacement=False, fraction=SAMPLE_SIZE, seed=SEED))
full_comments = (ss
                 .read
                 .options(**csv_options)
                 # .csv("dataset/nyt-comments-2020.csv")
                 .csv("dataset/nyt-comments-part0.csv")
                 .sample(withReplacement=False, fraction=SAMPLE_SIZE, seed=SEED)) # TODO: this is a subset of all comments

print(f"Number of articles: {full_articles.count()}")
print(f"Number of comments: {full_comments.count()}")
full_articles.take(1), full_comments.take(1)


Number of articles: 16787
Number of comments: 500000


([Row(newsdesk='Editorial', section='Opinion', subsection=None, material='Editorial', headline='Protect Veterans From Fraud', abstract='Congress could do much more to protect Americans who have served their country from predatory for-profit colleges.', keywords="['Veterans', 'For-Profit Schools', 'Financial Aid (Education)', 'Frauds and Swindling', 'Colleges and Universities', 'Veterans Affairs Department', 'Federal Trade Commission', 'University of Phoenix', 'Career Education Corporation']", word_count=680, pub_date=datetime.datetime(2020, 1, 1, 0, 18, 54), n_comments=186, uniqueID='nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')],
 [Row(commentID=104387472, status='approved', commentSequence=104387472, userID=60215558, userDisplayName='magicisnotreal', userLocation='earth', userTitle=None, commentBody='Here is something I think is fraudulent that vets are subject to\n If you use your VA home loan option you have to pay higher interest rates regardless of your credit rating becua

In [5]:
%pip install mmh3 # non-cryptographic hash, better distribution
# utility hash function: bytes | str -> signed int 32
def h(data):
    import mmh3
    return mmh3.hash(data, seed=0 if SEED is None else SEED)


Note: you may need to restart the kernel to use updated packages.


## RDD

In [ ]:
# parse keywords column utility
def parse_keywords(row):
    import re
    if not row["keywords"]: return []
    kws = re.sub(r'\[|\]|\'|\"', '', row["keywords"].lower()).strip()
    if not kws: return []
    return [w.strip() for w in kws.split(",") if len(w.strip()) > 0]

# keep only relevant information for recommendations porpouse, convert from dataframe to RDD tuples
articles = (full_articles
            .select("uniqueID", "newsdesk", "section", "subsection", "keywords")
            .rdd
            .map(lambda row: (row["uniqueID"], row["newsdesk"], row["section"], row["subsection"], parse_keywords(row))))
comments = (full_comments
            .select("articleID", "userID")
            .rdd
            .map(lambda row: (row["articleID"], row["userID"])))
users = (comments
         .map(lambda x: x[1])
         .distinct())

print(f"Number of articles: {articles.count()}")
print(f"Number of comments: {comments.count()}")
print(f"Number of users: {users.count()}")
articles.take(1), comments.take(1)


Number of articles: 16787
Number of comments: 500000
Number of users: 91437


([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   'Editorial',
   'Opinion',
   None,
   ['veterans',
    'for-profit schools',
    'financial aid (education)',
    'frauds and swindling',
    'colleges and universities',
    'veterans affairs department',
    'federal trade commission',
    'university of phoenix',
    'career education corporation'])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558)])

# Building Profiles

## Utility Matrix

In [ ]:
utility_matrix = (comments
                  .distinct()) # ignore multiple comments from same user on same article

print(f"Number of entries in sparse utility matrix: {utility_matrix.count()}")
utility_matrix.take(5)


Number of entries in sparse utility matrix: 356283


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65691034),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65110053),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 66232009),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 73299044)]

## Articles Profile - via dictionary

In [ ]:
# count frequencies
keywords = (articles
            .flatMap(lambda row: [(k, 1) for k in row[4]])
            .reduceByKey(lambda a, b: a + b))

# prune too rare keywords
pruned_keywords = (keywords
                   .filter(lambda x: x[1] >= MIN_KW_FREQ)
                   .map(lambda x: f"KEY {x[0]}")
                   .collect())

# extract all features
features = pruned_keywords + (articles
                              .flatMap(lambda row: [f"{f} {row[i+1].lower().strip()}" for i, f in enumerate(["NEW", "SEC", "SUB"]) if row[i+1]])
                              .distinct()
                              .collect())

# features mapping dictionary, needs to be broadcasted to every node
features_mapping = {f: i for i, f in enumerate(features)}
print(f"Extacted {len(features_mapping)} features (broadcasting ~{((100 * 4) + 4) * len(features_mapping) // 1024} KB)")
broadcast_dict = sc.broadcast(features_mapping)

# build sparse vectors for profiles (article_id, feature_index)
def build_article_profile_dict(row):
    mapping = broadcast_dict.value
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        if f not in mapping: continue
        res.append((row[0], mapping[f]))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        if k not in mapping: continue
        res.append((row[0], mapping[k]))
    return res

articles_profile_dict = (articles
                         .flatMap(build_article_profile_dict) # (article_id, feature_index)
                         .map(lambda x: (x[0], (x[1], 1)))) # (article_id, (feature_index, frequency = 1))
print(f"Number of entries in sparse articles profile via dictionary: {articles_profile_dict.count()}")
articles_profile_dict.take(10)


Extacted 3555 features (broadcasting ~1402 KB)
Number of entries in sparse articles profile via dictionary: 158079


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3384, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3385, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (0, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (4, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (5, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (6, 1)),
 ('nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c', (3386, 1))]

## Articles Profile - via hashing

In [ ]:
# build sparse vectors for profiles (article_id, feature_hash)
def build_article_profile_hash(row):
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        res.append((row[0], h(f) % FEATURES_SIZE))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        res.append((row[0], h(k) % FEATURES_SIZE))
    return res

articles_profile_hash = (articles
                         .flatMap(build_article_profile_hash) # (article_id, feature_hash)
                         .map(lambda x: (x[0], (x[1], 1)))) # (article_id, (feature_hash, frequency = 1))

print(f"Number of entries in sparse articles profile via hash: {articles_profile_hash.count()}")
articles_profile_hash.take(10)


Number of entries in sparse articles profile via hash: 185396


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (592, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (4811, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (4738, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2235, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (75, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (4154, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2938, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1131, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (4069, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (686, 1))]

## WARNING

From now on, the operations will be **independent** of how the profiles were computed (via dictionary or hashing).
Change the content of next cell to change approach: (`articles_profile_dict` or `articles_profile_hash`)

In [ ]:
# articles_profile = articles_profile_dict
articles_profile = articles_profile_hash


PythonRDD[74] at RDD at PythonRDD.scala:53

## Users Profile

In [ ]:
# build user profiles by joining utility matrix with article profiles, then counting feature frequencies for each user
users_profile = (utility_matrix
                 .join(articles_profile) # (article_id, (user_id, (feature_index, frequency = 1)))
                 .map(lambda x: ((x[1][0], x[1][1][0]), 1)) # ((user_id, feature_index), 1)
                 .reduceByKey(lambda a, b: a + b) # ((user_id, feature_index), frequency)
                 .map(lambda x: (x[0][0], (x[0][1], x[1])))) # (user_id, (feature_index, frequency))
print(f"Number of entries in sparse users profile: {users_profile.count()}")
users_profile.take(10)


Number of entries in sparse users profile: 2818910


[(5646097, (4379, 36)),
 (5646097, (1731, 36)),
 (86273619, (4379, 11)),
 (86273619, (1731, 11)),
 (61536727, (4379, 9)),
 (61536727, (1731, 9)),
 (67789834, (2436, 22)),
 (41508509, (4379, 20)),
 (41508509, (1731, 20)),
 (29604532, (2436, 11))]

# Computing Similarity

## Profiles sketches (signatures)

In [ ]:
import random
if SEED is not None: random.seed(SEED)

# random vectors of +1 and -1 of dimensione FEATURES_SIZE
random_hyperplanes = [[random.choice([-1, 1]) for _ in range(FEATURES_SIZE)] for _ in range(LSH_NUM_SKETCHES)]
b_hyperplanes = sc.broadcast(random_hyperplanes)
print(f"Generated {LSH_NUM_SKETCHES} random hyperplanes (broadcasting ~{(LSH_NUM_SKETCHES * FEATURES_SIZE * 4) // 1024} KB)")

def compute_sketch(row):
    item_id, sparse_features = row
    sketch = []
    for plane in b_hyperplanes.value:
        # dot product: if above or below the hyperplane
        # only considering the features that exist in the sparse vector
        dot_product = sum(weight * plane[feat] for feat, weight in sparse_features)
        # 1 if positive, 0 if negative
        sketch.append(1 if dot_product > 0 else 0)
    return (item_id, sketch)

# compute sketches ("equivalent" of signature) for users and articles
users_sketches = (users_profile # (user_id, (feature_index, frequency))
                  .groupByKey()
                  .mapValues(list) # (user_id, [(feature_index, frequency), ...])
                  .map(compute_sketch)) # (user_id, sketch)
articles_sketches = (articles_profile # (article_id, (feature_index, frequency = 1))
                     .groupByKey()
                     .mapValues(list) # (article_id, [(feature_index, frequency), ...])
                     .map(compute_sketch)) # (article_id, sketch)

print(f"Number of users sketches: {users_sketches.count()}")
print(f"Number of articles sketches: {articles_sketches.count()}")
users_sketches.take(1), articles_sketches.take(1)


Generated 100 random hyperplanes (broadcasting ~1953 KB)
Number of users sketches: 91437
Number of articles sketches: 16787


([(67789834,
   [0,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    0])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   [1,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
   

## LSH buckets

In [ ]:
def lsh_buckets(row):
    item_id, sketch = row
    res = []
    for band in range(LSH_BANDS):
        band_signature = tuple(sketch[band*LSH_ROWS : (band+1)*LSH_ROWS])
        bucket_hash = h(f"b{band}_{band_signature}") # include band index to avoid collisions between different bands
        res.append((bucket_hash, item_id)) # no need to differentiate between users and articles: different format and distinct rdds
    return res

users_buckets = users_sketches.flatMap(lsh_buckets) # (bucket_hash, user_id)
articles_buckets = articles_sketches.flatMap(lsh_buckets) # (bucket_hash, article_id)

print(f"Total number of users buckets: {users_buckets.count()}")
print(f"Total number of articles buckets: {articles_buckets.count()}")

# buckets too big are not useful, too few information
valid_users_buckets = (users_buckets
                       .map(lambda x: (x[0], 1))
                       .reduceByKey(lambda a, b: a + b)
                       .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

valid_articles_buckets = (articles_buckets
                          .map(lambda x: (x[0], 1))
                          .reduceByKey(lambda a, b: a + b)
                          .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

# the join works as an intersection between keys
users_buckets = (users_buckets # (bucket_hash, user_id)
                 .join(valid_users_buckets) # (bucket_hash, (user_id, count))
                 .map(lambda x: (x[0], x[1][0]))) # (bucket_hash, user_id)
articles_buckets = (articles_buckets # (bucket_hash, article_id)
                    .join(valid_articles_buckets) # (bucket_hash, (article_id, count))
                    .map(lambda x: (x[0], x[1][0]))) # (bucket_hash, article_id)

print(f"Valid (less than {LSH_MAX_BUCKET_SIZE} users) number of users buckets: {users_buckets.count()} ")
print(f"Valid (less than {LSH_MAX_BUCKET_SIZE} articles) number of articles buckets: {articles_buckets.count()} ")

users_buckets.take(10), articles_buckets.take(10)


Total number of users buckets: 914370
Total number of articles buckets: 167870
Valid (less than 500 users) number of users buckets: 549068 
Valid (less than 500 articles) number of articles buckets: 161078 


([(2009018280, 228594),
  (2009018280, 89363406),
  (2009018280, 14918140),
  (2009018280, 55582500),
  (2009018280, 73491210),
  (2009018280, 66081278),
  (2009018280, 74175040),
  (2009018280, 41914976),
  (2009018280, 107818706),
  (2009018280, 59687022)],
 [(1991176582, 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd'),
  (1991176582, 'nyt://article/b496c678-15f2-5f3a-8d12-66bcfe73fd20'),
  (1991176582, 'nyt://article/f8add4e8-4e48-5ce8-b6b9-865f6ef0a0dc'),
  (1991176582, 'nyt://article/866f0de9-b801-54fb-b719-47a64a32e213'),
  (1991176582, 'nyt://article/1a98dc7a-0a4a-5a1f-b78a-5038d99d4273'),
  (1991176582, 'nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b'),
  (1991176582, 'nyt://article/5556cbcb-873f-5af5-a27c-7e7488f9040e'),
  (1991176582, 'nyt://article/3e731962-9b8b-50f1-bede-47fed0698e1a'),
  (1991176582, 'nyt://article/7cb14fca-4cd7-5a9f-a4f4-a13734ed3428'),
  (1991176582, 'nyt://article/9bbcf3b0-39f2-52e6-a15d-53ccdd3c411c')])

## LSH filter

In [ ]:
lsh_recommendations = (users_buckets # (bucket_hash, user_id)
                       .join(articles_buckets) # (bucket_hash, (user_id, article_id))
                       .map(lambda x: (x[1][0], x[1][1])) # (user_id, article_id)
                       .distinct() # remove matches from multiple bands
                       .subtract(utility_matrix.map(lambda x: (x[1], x[0])))) # remove already commented articles

print(f"Number of recommendations that passed LSH filter: {lsh_recommendations.count()}")
lsh_recommendations.take(10)


Number of recommendations that passed LSH filter: 12483493


[(148372, 'nyt://article/a8895d67-845f-5333-9faf-57722a048c94'),
 (78142584, 'nyt://article/8b58d49c-ce9a-578a-8304-d6a5c9ac7977'),
 (30395127, 'nyt://article/74d8b7b8-b644-52fe-9f2a-d08b99885347'),
 (44696946, 'nyt://article/cd755fa9-38a7-56a1-8b8e-5bd28eddc0d2'),
 (78095413, 'nyt://article/e4293b44-d440-52cc-b3cc-e2f6ae8d335a'),
 (63705790, 'nyt://article/fa9e9c8b-c7d7-523f-bd7a-f937d18b715f'),
 (66402226, 'nyt://article/243343a3-ef2c-56e3-b6b5-4298f31a9437'),
 (60910799, 'nyt://article/2088f10a-46eb-581e-b288-b5ed669ad53e'),
 (58932858, 'nyt://article/735b0d87-e3d9-5ece-b248-04d9c78f0b31'),
 (67628195, 'nyt://article/10ff5205-6442-5416-98fc-00d4ba687f9e')]

## Actual Cosine Similarity

In [ ]:
# cosine similarity between sparse profiles, user_profile and article_profile should be dicts {feature_index: weight}
def cosine_similarity(user_profile, article_profile):
    dot_product = sum(user_profile.get(feat, 0) * weight for feat, weight in article_profile.items())
    user_norm = sum(weight ** 2 for weight in user_profile.values()) ** 0.5
    article_norm = sum(weight ** 2 for weight in article_profile.values()) ** 0.5
    if user_norm == 0 or article_norm == 0:
        return 0.0
    return dot_product / (user_norm * article_norm)

# calculate actual similarity by joining the profiles and applying
recommendations = (lsh_recommendations # (user_id, article_id)
                   .join(users_profile.groupByKey().mapValues(dict)) # (user_id, (article_id, {profile}))
                   .map(lambda x: (x[1][0], (x[0], x[1][1]))) # (article_id, (user_id, {user_profile}))
                   .join(articles_profile.groupByKey().mapValues(dict)) # (article_id, ((user_id, {user_profile}), {article_profile}))
                   .map(lambda x: (x[1][0][0], (x[0], cosine_similarity(x[1][0][1], x[1][1])))) # (user_id, (article_id, actual_sim))
                   .filter(lambda x: x[1][1] > 0.0))

print(f"Number of recommendations with actual cosine similarity: {recommendations.count()}")
recommendations.take(10)


Number of recommendations with actual cosine similarity: 6171001


[(65172240,
  ('nyt://article/f5eb779a-d763-5d59-9cb0-062a2b18f8f1', 0.09759000729485333)),
 (105219729,
  ('nyt://article/f5eb779a-d763-5d59-9cb0-062a2b18f8f1', 0.14142135623730948)),
 (69438600,
  ('nyt://article/f5eb779a-d763-5d59-9cb0-062a2b18f8f1', 0.14142135623730948)),
 (85949001,
  ('nyt://article/f5eb779a-d763-5d59-9cb0-062a2b18f8f1',
   0.058722021951470346)),
 (100066239,
  ('nyt://article/f5eb779a-d763-5d59-9cb0-062a2b18f8f1', 0.14142135623730948)),
 (61584444,
  ('nyt://article/f5eb779a-d763-5d59-9cb0-062a2b18f8f1', 0.14142135623730948)),
 (61475607,
  ('nyt://article/f5eb779a-d763-5d59-9cb0-062a2b18f8f1', 0.14142135623730948)),
 (92933244,
  ('nyt://article/f5eb779a-d763-5d59-9cb0-062a2b18f8f1', 0.14142135623730948)),
 (52544979,
  ('nyt://article/f5eb779a-d763-5d59-9cb0-062a2b18f8f1', 0.14142135623730948)),
 (92643966,
  ('nyt://article/f5eb779a-d763-5d59-9cb0-062a2b18f8f1', 0.14142135623730948))]

# Content-Based Recommendations

In [ ]:
import heapq

user_recommendations = (recommendations
                        .map(lambda x: (x[0], (x[1][0], round(x[1][1], 5))))
                        .groupByKey()
                        .mapValues(lambda iter: heapq.nlargest(RECOMMENDED_ARTICLES, iter, key=lambda x: x[1])))

print(f"Number of users with at least one recommendation: {user_recommendations.count()}/{users.count()}")
user_recommendations.take(10)


Number of users with at least one recommendation: 83601/91437


[(90258570,
  [('nyt://article/6bc0b816-6248-5131-a4f2-cee9fae576f3', 0.75703),
   ('nyt://article/f90d25e2-9b88-5c17-b980-64d84599cb08', 0.7456),
   ('nyt://article/aa7dd152-2127-50f1-88cf-e49b82e3ae7f', 0.7456),
   ('nyt://article/5c8fdeb8-9ec4-5ad7-b5a3-d6d22504352e', 0.7456),
   ('nyt://article/5e62ca8f-ae51-5c14-8b93-7a22635ef49c', 0.71818),
   ('nyt://article/0457a10e-6ae9-584f-aa93-e49474764a7e', 0.68825),
   ('nyt://article/353d9280-590d-5a72-b706-960a9abc66a0', 0.65561),
   ('nyt://article/b78d1a33-0721-5928-a25e-fccf133a7887', 0.64889),
   ('nyt://article/1d487a21-8e74-589c-aaba-7dd62396c794', 0.64889),
   ('nyt://article/69f5e9c2-d0ce-53a4-b73e-22a661e86d78', 0.63089)]),
 (55531710,
  [('nyt://article/efd88313-6c28-58d8-9608-94af430623ac', 0.70711),
   ('nyt://article/4b4523f9-d7c0-5a50-925e-16338fa0f04f', 0.61237),
   ('nyt://article/763e75bc-5bff-549a-9389-f1a4b8ea135e', 0.61237),
   ('nyt://article/ebb8906b-bdd2-5e64-963f-302cc7890912', 0.61237),
   ('nyt://article/5340c1

## Human Evaluation

In [17]:
USER = users.takeSample(False, 1, seed=SEED)[0]

# "notable" users:
# USER = 65969500 # bad results
# USER = 22564580 # policits but not trump

read = (utility_matrix
        .filter(lambda x: x[1] == USER)
        .join(full_articles.rdd.map(lambda x: (x['uniqueID'], x.asDict()))) # (article_id, (user_id, article_info))
        .map(lambda x: x[1][1]) # (article_info)
        .collect())

reccs = (user_recommendations
         .filter(lambda x: x[0] == USER)
         .map(lambda x: x[1])
         .flatMap(lambda x: x)
         .join(full_articles.rdd.map(lambda x: (x['uniqueID'], x.asDict()))) # (article_id, article_info)
         .map(lambda x: (x[1][1], x[1][0])) # (article_info, similarity)
         .collect())

print("--- ARTICLES COMMENTED ---")
for r in read:
    kws = "\n\t".join(parse_keywords(r))
    print(f"  {r['headline']}\n\t{r['newsdesk']} - {r['section']} - {r['subsection']}\n\t{kws}")

print("--- ARTICLES RECOMMENDED ---")
for r, s in sorted(reccs, key=lambda x: x[1], reverse=True):
    kws = "\n\t".join(parse_keywords(r))
    print(f"  {round(s, 5)} {r['headline']}\n\t{r['newsdesk']} - {r['section']} - {r['subsection']}\n\t{kws}")


--- ARTICLES COMMENTED ---
  Biden, Sanders, Social Security and Smears
	OpEd - Opinion - None
	presidential election of 2020
	united states politics and government
	social security (us)
	democratic party
	biden
	joseph r jr
	sanders
	bernard
	trump
	donald j
  If Bernie Wins, Where Will He Take the Democratic Party?
	OpEd - Opinion - None
	presidential election of 2020
	presidential election of 2016
	primaries and caucuses
	midterm elections (2018)
	united states politics and government
	democratic party
	buttigieg
	pete (1982- )
	chait
	jonathan
	biden
	joseph r jr
	french
	david a
	sanders
	bernard
	trump
	donald j
--- ARTICLES RECOMMENDED ---
  0.78567 Have We Learned Nothing After Four Years of Trump?
	OpEd - Opinion - None
	presidential election of 2020
	conservatism (us politics)
	democratic party
	republican party
	biden
	joseph r jr
	trump
	donald j
	sanders
	bernard
  0.6931 I Asked Bernie Sanders if It Was All Over. ‘No,’ He Groaned.
	OpEd - Opinion - None
	sanders
	bernard
